# 1. Exploring the loans (weeks 2-4)
**Question:** which borrowers fail to repay, and does LendingClub's interest rate follow that risk?

Each chart below answers one small question. For each, write one sentence of your own *before* reading the comment under it.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import plotly.express as px
from lc_rates import config, data, features, finance, model
pd.options.display.float_format = "{:,.4f}".format

## The clean table
Only **finished** loans (Fully Paid or Charged Off) that have had **time to finish**. Look at the columns: which ones are known when a person applies, and which only appear after the loan has run?

In [2]:
df = data.load_clean()
print(df.shape)
df.head()

(673314, 29)


,id,issue_date,issue_year,term_months,loan_amnt,annual_inc,dti,fico_mid,emp_length_years,home_ownership,...,credit_history_years,grade,sub_grade,int_rate,funded_amnt,total_pymnt,total_rec_prncp,recoveries,months_paid,defaulted
0,68407277,2015-12-01,2015,36,"3,600.0000","55,000.0000",5.9100,677.0000,10.0000,MORTGAGE,...,12.3340,C,C4,13.9900,"3,600.0000","4,421.7239","3,600.0000",0.0000,36.0000,0
1,68355089,2015-12-01,2015,36,"24,700.0000","65,000.0000",16.0600,717.0000,10.0000,MORTGAGE,...,16.0000,C,C1,11.9900,"24,700.0000","25,679.6600","24,700.0000",0.0000,6.0000,0
2,68426831,2015-12-01,2015,36,"11,950.0000","34,000.0000",10.2000,692.0000,4.0000,RENT,...,28.1670,C,C3,13.4400,"11,950.0000","13,708.9485","11,950.0000",0.0000,17.0000,0
3,68476668,2015-12-01,2015,36,"20,000.0000","180,000.0000",14.6700,682.0000,10.0000,MORTGAGE,...,25.5003,B,B2,9.1700,"20,000.0000","21,393.8000","20,000.0000",0.0000,11.0000,0
4,67275481,2015-12-01,2015,36,"20,000.0000","85,000.0000",17.6100,707.0000,10.0000,MORTGAGE,...,16.8296,B,B1,8.4900,"20,000.0000","21,538.5090","20,000.0000",0.0000,13.0000,0


In [3]:
df.describe().T[['count', 'mean', 'min', 'max']]

,count,mean,min,max
id,"673,314.0000","29,328,865.1933","54,734.0000","68,616,919.0000"
issue_date,673314,2014-05-07 06:34:05.364866,2007-06-01 00:00:00,2015-12-01 00:00:00
issue_year,"673,314.0000","2,013.8444","2,007.0000","2,015.0000"
term_months,"673,314.0000",37.9593,36.0000,60.0000
loan_amnt,"673,314.0000","13,091.2937",500.0000,"35,000.0000"
annual_inc,"673,314.0000","73,073.6085","3,000.0000","9,000,000.0000"
dti,"673,314.0000",17.5467,0.0000,39.9900
fico_mid,"673,314.0000",697.5222,627.0000,847.5000
emp_length_years,"635,784.0000",5.8991,0.0000,10.0000
delinq_2yrs,"673,314.0000",0.3082,0.0000,30.0000


## Default rate by LendingClub grade

In [4]:
g = df.groupby('grade').agg(loans=('defaulted', 'size'), default_rate=('defaulted', 'mean'), interest_rate=('int_rate', 'mean')).reset_index()
fig = px.bar(g, x='grade', y='default_rate', text=g['default_rate'].map('{:.1%}'.format), title='Share of loans that defaulted, by grade')
fig.update_layout(yaxis_tickformat='.0%')
fig.show()
g

,grade,loans,default_rate,interest_rate
0,A,144098,0.0550,7.2248
1,B,219762,0.1136,10.8970
2,C,179506,0.1829,14.0822
3,D,86392,0.2380,17.3128
4,E,31592,0.2941,20.1991
5,F,10099,0.3444,23.4021
6,G,1865,0.3748,24.8635


Default rises steadily from A to G, so the grade carries real information. The next question is whether the interest rate rises *enough*.

In [5]:
df['fico_band'] = pd.cut(df['fico_mid'], [0, 680, 700, 720, 740, 760, 900], right=False, labels=['<680', '680-699', '700-719', '720-739', '740-759', '760+'])
df['income_quartile'] = pd.qcut(df['annual_inc'], 4, labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'])
for col in ['fico_band', 'income_quartile', 'purpose']:
    t = df.groupby(col, observed=True)['defaulted'].mean().reset_index()
    px.bar(t, x=col, y='defaulted', title=f'Default rate by {col}').update_layout(yaxis_tickformat='.0%').show()

## The maturity trap
Many loans in the data are still being repaid ("Current"). Whatever you do with them distorts the default rate. Compare three ways of measuring it, by the year the loan was issued.

In [6]:
raw = pd.read_csv(config.ACCEPTED_RAW, usecols=['loan_status', 'issue_d']).dropna()
raw['year'] = raw['issue_d'].str[-4:].astype(int)
everyone = raw.assign(d=raw['loan_status'] == 'Charged Off').groupby('year')['d'].mean()
done = raw[raw['loan_status'].isin(['Fully Paid', 'Charged Off'])]
finished = done.assign(d=done['loan_status'] == 'Charged Off').groupby('year')['d'].mean()
matured = df.groupby('issue_year')['defaulted'].mean()
trap = pd.concat({'A. every loan, Current counted as fine': everyone, 'B. finished loans, no maturity rule': finished,
                  'C. only loans old enough to be finished (ours)': matured}, axis=1)
trap.index.name = 'year'
px.line(trap.reset_index().melt('year', var_name='method', value_name='default_rate'), x='year', y='default_rate', color='method', markers=True,
        title='Default rate by issue year, three ways').update_layout(yaxis_tickformat='.0%', legend_title='').show()
trap.round(3).T

year,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
"A. every loan, Current counted as fine",0.0750,0.1030,0.1120,0.1190,0.1520,0.1620,0.1560,0.1750,0.1800,0.1570,0.0880,0.0180
"B. finished loans, no maturity rule",0.1790,0.1580,0.1260,0.1290,0.1520,0.1620,0.1560,0.1840,0.2020,0.2330,0.2310,0.1570
C. only loans old enough to be finished (ours),0.1790,0.1580,0.1260,0.1290,0.1520,0.1620,0.1560,0.1370,0.1490,NaN,NaN,NaN


* **A** makes 2017-18 look very safe (1.8% in 2018) only because those loans have had no time to default.
* **B** throws away the "Current" loans, but loans that go bad are resolved early while healthy ones are still running, so the finished pool is full of defaults: 2016 looks like 23%. It also mixes 36- and 60-month loans, and 60-month loans default more.
* **C** keeps only loans whose last payment was due by the end of 2018. This is the only fair comparison. That is why the 2014-15 test set contains 36-month loans only.

Neither mistake raises an error: the code runs and the chart looks plausible. This is the kind of mistake that makes a good 'what I got wrong' paragraph in an essay.

### Your turn
1. Which purpose has the highest default rate? Does it surprise you?
2. Is the 60-month default rate higher than the 36-month one? Why might that be?
3. Pick any two columns and find a relationship we did not chart.